# IGNIS — Next-Day Wildfire Spread Dataset Generation (v3)
# IGNIS — Ertesi-Gün Yangın Yayılım Veri Seti Üretimi

**Paper / Bildiri:** `IAC-26,B1,IP,107,x110901` — 77th International Astronautical Congress,
Antalya, Türkiye, 5–9 October 2026.
**Repository:** `github.com/WhiteFoxYT/ignis-ai`

---

## EN — What this notebook does

This notebook builds the supervised training archive for the IGNIS next-day wildfire
spread model. For every day on which Türkiye had significant active fire activity, it

1. locates burning pixels from the merged MODIS Terra + Aqua thermal anomaly products,
2. assembles a 17-band raster stack of environmental drivers on a common 1 km UTM 35N grid,
3. extracts a 65 × 65 pixel neighbourhood around each sampled fire pixel, and
4. exports those patches to Google Drive as gzip-compressed `TFRecord` shards, one per day.

## TR — Bu notebook ne yapar

Bu notebook, IGNIS ertesi-gün yangın yayılım modelinin denetimli eğitim arşivini üretir.
Türkiye'de kayda değer aktif yangın görülen her gün için

1. MODIS Terra + Aqua termal anomali ürünlerinden yanan pikselleri bulur,
2. ortak 1 km UTM 35N gridinde 17 bantlı bir çevresel sürücü yığını oluşturur,
3. örneklenen her yangın pikselinin etrafından 65 × 65 piksellik bir komşuluk çıkarır,
4. bu yamaları Google Drive'a günlük `TFRecord` parçaları olarak yazar.

---

## Changes in this revision / Bu sürümdeki değişiklikler

Three defects in the previous revision were identified by direct measurement of the
exported archive (45 shards / 1054 patches sampled). This revision corrects them.

Önceki sürümdeki üç kusur, dışa aktarılmış arşivin doğrudan ölçülmesiyle tespit edildi
(45 parça / 1054 kare örneklendi). Bu sürüm onları düzeltir.

| # | Problem | Correction / Düzeltme |
|---|---|---|
| 1 | 58.9 % of patches contained **no fire pixel at all** on day *t*+1, while 12.3 pixels burned on average on day *t*. At 1 km resolution the next-day mask largely encodes cloud cover, smoke opacity and orbital timing rather than fire physics. | New band **`fire_next2`** (day *t*+2). Training uses `target = max(fire_next, fire_next2)`, i.e. **fire activity within the next 24–48 h**. The strict day *t*+1 mask is retained unchanged so both target definitions remain available. |
| 2 | An identical **~15 % zero rate** was measured across every environmental band — an artefact of `clip(REGION)` followed by `unmask(0)`. The model could not distinguish "relative humidity = 0 %" from "no observation". | New band **`valid`**, computed from the band masks *before* `unmask`. It is supplied to the network as an input channel and used to mask the loss. |
| 3 | The archive ended on **26 July 2021**, two days before the Manavgat and Marmaris fires — the largest wildfire event in modern Turkish history — began. | Coverage extended to **2019–2026**, with the final date detected automatically from collection metadata. |

Additional operational improvements / Ek işletme iyileştirmeleri:

- **Resumable.** Days already present in Drive are skipped, so an interrupted Colab
  session can simply be re-run. / Drive'da hâlihazırda bulunan günler atlanır.
- **Fast day selection.** Fire-pixel counts are computed server-side one month at a time
  (≈40 round trips instead of ≈1100). / Yangın piksel sayımları sunucu tarafında aylık
  gruplar hâlinde hesaplanır.
- **Precipitation fallback.** CHIRPS lags real time by several weeks; ERA5-Land
  `total_precipitation_sum` is substituted when CHIRPS is unavailable. / CHIRPS'in
  gecikmesi olduğunda ERA5-Land yağışı kullanılır.
- **Meteorological fallback.** The most recent ERA5-Land record within a 7-day window is
  used if the exact date is not yet published. / Tam tarih henüz yayımlanmadıysa 7 günlük
  pencere içindeki en yeni ERA5-Land kaydı kullanılır.

---

## v3 — additional channels / ek kanallar

v3 extends the v2 band contract from 14 to 19 input channels. Everything else —
grid, patch geometry, sampling, target definition, outage handling — is unchanged,
so a v3 archive is a strict superset of a v2 archive.

v3, v2 bant sözleşmesini 14 girdi kanalından 19'a çıkarır. Grid, yama geometrisi,
örnekleme, hedef tanımı ve kesinti yönetimi dâhil geri kalan her şey aynıdır; bu
nedenle bir v3 arşivi, bir v2 arşivinin tam üst kümesidir.

| New channel | Rationale / Gerekçe |
|---|---|
| `fire_prev1`, `fire_prev2` | A one-day snapshot cannot separate a fire burning continuously for three days from an isolated thermal anomaly, yet the two evolve very differently. Two published treatments of this problem — Convolutional LSTM for wildland fire dynamics, and CNN-BiLSTM for near-real-time daily spread (*Remote Sensing* 16(8):1467) — both resolve it with temporal context. Adding *t*−1 and *t*−2 as channels captures most of that benefit without a recurrent architecture. |
| `vpd` | Vapour pressure deficit is a more direct proxy for fine-fuel moisture than relative humidity, because identical RH is far more desiccating at high temperature. Standard in operational fire-weather indices. |
| `precip_7d`, `precip_30d` | The 24-hour precipitation channel is zero on **91.5 %** of pixels in the v2 archive and so carries almost no information alone. Accumulated antecedent precipitation encodes the drought state that governs fuel dryness. |

**Exports to a separate Drive folder (`GEE_FireSpread_v3`).** The v2 notebook and its
archive are left untouched and remain reproducible; train on v2 while v3 is being
generated, then retrain and compare.

**Ayrı bir Drive klasörüne yazar (`GEE_FireSpread_v3`).** v2 notebook'u ve arşivi
olduğu gibi kalır ve yeniden üretilebilir olmayı sürdürür; v3 üretilirken v2 ile
eğitin, sonra yeniden eğitip karşılaştırın.


## 1 — Environment / Ortam

In [ ]:
# Colab: run once per session. / Colab: oturum başına bir kez çalıştırın.
!pip install -q earthengine-api geemap

In [ ]:
import datetime
import ee
import geemap

# Replace with your own Earth Engine project ID.
# Kendi Earth Engine proje kimliğinizi yazın.
EE_PROJECT = 'ignisai-496207'

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print(f'Earth Engine ready / hazir  —  project: {EE_PROJECT}')

## 2 — Configuration / Konfigürasyon

Every constant that defines the dataset lives in this cell. The band order declared in
`INPUT_BANDS` is **contractual**: it must match `src/config.py → SPREAD_INPUT_BANDS` and
`src/gee_config.py → GEEConfig.INPUT_BANDS` exactly, because the training pipeline
reconstructs the channel axis from this order alone.

Veri setini tanımlayan tüm sabitler bu hücrededir. `INPUT_BANDS` içindeki bant sırası
**sözleşmeseldir**: `src/config.py → SPREAD_INPUT_BANDS` ve
`src/gee_config.py → GEEConfig.INPUT_BANDS` ile birebir aynı olmalıdır, çünkü eğitim
hattı kanal eksenini yalnızca bu sıraya bakarak yeniden kurar.

In [ ]:
# ─────────────────────── STUDY AREA / ÇALIŞMA ALANI ───────────────────────
REGION = (ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')
            .filter(ee.Filter.eq('country_na', 'Turkey')))

# Analysis grid: UTM zone 35N at 1 km. Metric and square over Türkiye.
# Analiz gridi: UTM 35N, 1 km. Türkiye üzerinde metrik ve kare.
SCALE = 1000                                   # metres per pixel / metre / piksel
PROJ  = ee.Projection('EPSG:32635').atScale(SCALE)

# ─────────────────────── PATCH GEOMETRY / YAMA GEOMETRİSİ ─────────────────
PATCH_RADIUS = 32                              # -> 65 x 65
PATCH_SIZE   = 2 * PATCH_RADIUS + 1

# ─────────────────────── TEMPORAL COVERAGE / ZAMANSAL KAPSAM ──────────────
FIRE_SEASON = (6, 10)                          # June - October / Haziran - Ekim
YEARS       = [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

# ─────────────────────── ACTIVE FIRE / AKTİF YANGIN ───────────────────────
# MODIS FireMask classes: 7 = low confidence, 8 = nominal, 9 = high.
# MODIS FireMask sınıfları: 7 = düşük güven, 8 = orta, 9 = yüksek.
FIRE_CONFIDENCE    = 7
MIN_FIRE_PIXELS    = 5      # minimum active fire pixels for a day to be exported
MAX_POINTS_PER_DAY = 150    # patches sampled per fire day / gün başına yama sayısı
SEED               = 42

# ─────────────────────── BAND CONTRACT / BANT SÖZLEŞMESİ ──────────────────
INPUT_BANDS = [
    'ndvi',            # 1  vegetation index          bitki örtüsü indeksi
    'lst',             # 2  land surface temp. [degC] arazi yüzey sıcaklığı
    'air_temp',        # 3  2 m air temp.     [degC]  hava sıcaklığı
    'humidity',        # 4  relative humidity [%]     bağıl nem
    'vpd',             # 5  vapour pressure deficit [kPa]  buhar basıncı açığı
    'wind_speed',      # 6  wind magnitude    [m/s]   rüzgâr hızı
    'wind_u',          # 7  eastward wind     [m/s]   rüzgâr doğu bileşeni
    'wind_v',          # 8  northward wind    [m/s]   rüzgâr kuzey bileşeni
    'precip',          # 9  precipitation, 24 h [mm]  yağış, 24 saat
    'precip_7d',       # 10 precipitation, 7 d  [mm]  birikimli yağış, 7 gün
    'precip_30d',      # 11 precipitation, 30 d [mm]  birikimli yağış, 30 gün
    'soil_moisture',   # 12 soil water        [m3/m3] toprak nemi
    'elevation',       # 13 elevation         [m]     yükseklik
    'slope',           # 14 slope             [deg]   eğim
    'aspect',          # 15 aspect            [deg]   bakı
    'landcover',       # 16 IGBP fuel class           yakıt sınıfı
    'fire_prev2',      # 17 fire mask, day t-2        iki gün önceki yangın maskesi
    'fire_prev1',      # 18 fire mask, day t-1        dünkü yangın maskesi
    'fire',            # 19 fire mask, day t          bugünün yangın maskesi
]
TARGET_BANDS = [
    'fire_next',       # fire mask, day t+1           yarının yangın maskesi
    'fire_next2',      # fire mask, day t+2           öbür günün yangın maskesi
    'valid',           # 1 where all inputs observed  tüm girdiler gözlendiyse 1
]
ALL_BANDS = INPUT_BANDS + TARGET_BANDS
META_COLS = ['lon', 'lat', 'date']

# ─────────────────────── GROWTH CLASSES / BÜYÜME SINIFLARI ────────────────
# r = N(t+1) / max(N(t), 1)
GROW_RATIO = 1.25          # r > 1.25              -> growing    / büyüyor
STABLE_LOW = 0.75          # 0.75 <= r <= 1.25     -> stable     / sabit
#                            r < 0.75              -> extinguishing / sönüyor

# ─────────────────────── EXPORT / DIŞA AKTARIM ────────────────────────────
# NOTE: a NEW folder. The previous archive in `GEE_FireSpread` uses the old 15-band
# schema and lacks `fire_next2` and `valid`; mixing the two would silently corrupt
# training. Writing to a separate folder also lets the resume scan work correctly,
# because otherwise the 2019-2021 shards already in Drive would be skipped -- exactly
# the days that must be regenerated.
# NOT: YENI bir klasor. `GEE_FireSpread` icindeki onceki arsiv eski 15 bantli semayi
# kullanir ve `fire_next2` ile `valid` bantlarini icermez; ikisini karistirmak egitimi
# sessizce bozar. Ayri klasore yazmak ayrica devam-etme taramasinin dogru calismasini
# saglar; aksi halde Drive'da zaten bulunan 2019-2021 parcalari atlanirdi -- ki tam da
# yeniden uretilmesi gereken gunler onlardir.
DRIVE_FOLDER = 'GEE_FireSpread_v3'
SUBMIT_LIMIT = 400   # export tasks submitted per notebook run; re-run to continue
                     # her çalıştırmada açılacak görev sayısı; devam için tekrar çalıştırın

print(f'Patch      : {PATCH_SIZE} x {PATCH_SIZE} px @ {SCALE} m')
print(f'Bands      : {len(INPUT_BANDS)} input + {len(TARGET_BANDS)} target = {len(ALL_BANDS)}')
print(f'Years      : {YEARS[0]}-{YEARS[-1]}, months {FIRE_SEASON[0]}-{FIRE_SEASON[1]}')

## 3 — Data availability / Veri erişilebilirliği

MODIS thermal anomaly products reach Earth Engine with a latency of a few days, and the
target definition requires day *t*+2. The last exportable sample date is therefore
`last(MOD14A1) − 2 days`. This cell queries the collections directly rather than assuming
a fixed end date, so the notebook remains correct as new data is published.

MODIS termal anomali ürünleri Earth Engine'e birkaç gün gecikmeyle ulaşır ve hedef tanımı
*t*+2 gününü gerektirir. Bu nedenle dışa aktarılabilir son örnek tarihi
`son(MOD14A1) − 2 gün`'dür. Bu hücre sabit bir bitiş tarihi varsaymak yerine koleksiyonları
doğrudan sorgular; böylece yeni veri yayımlandıkça notebook doğru kalmaya devam eder.

In [ ]:
def collection_span(cid):
    '''Return (first, last) acquisition dates of a collection as ISO strings.'''
    c = ee.ImageCollection(cid)
    lo = ee.Date(c.aggregate_min('system:time_start')).format('YYYY-MM-dd')
    hi = ee.Date(c.aggregate_max('system:time_start')).format('YYYY-MM-dd')
    return ee.List([lo, hi]).getInfo()


CHECK = [
    ('MODIS/061/MOD14A1',            'active fire (Terra)'),
    ('MODIS/061/MYD14A1',            'active fire (Aqua)'),
    ('MODIS/061/MOD13Q1',            'NDVI'),
    ('MODIS/061/MOD11A1',            'land surface temperature'),
    ('ECMWF/ERA5_LAND/DAILY_AGGR',   'meteorology'),
    ('UCSB-CHG/CHIRPS/DAILY',        'precipitation'),
    ('MODIS/061/MCD12Q1',            'land cover'),
]

print(f'{"collection":<32}{"purpose":<28}{"first":<12}{"last":<12}')
print('-' * 84)
spans = {}
for cid, purpose in CHECK:
    lo, hi = collection_span(cid)
    spans[cid] = (lo, hi)
    print(f'{cid:<32}{purpose:<28}{lo:<12}{hi:<12}')

# Latest sample date we can build a t+2 target for.
# t+2 hedefi kurabileceğimiz en son örnek tarihi.
_last_fire = datetime.date.fromisoformat(spans['MODIS/061/MOD14A1'][1])
LAST_SAMPLE_DATE = _last_fire - datetime.timedelta(days=2)
print(f'\nLast usable sample date / son kullanilabilir ornek tarihi: {LAST_SAMPLE_DATE}')

## 4 — Raster construction / Raster kurulumu

### Fire mask / Yangın maskesi

Terra and Aqua cross the equator at different local times, so merging both `FireMask`
bands with a pixel-wise maximum roughly doubles the daily detection opportunity. Pixels
whose confidence class is below `FIRE_CONFIDENCE` — including the water, cloud and
not-processed classes — are treated as non-fire.

Terra ve Aqua ekvatoru farklı yerel saatlerde geçer; bu yüzden iki `FireMask` bandını
piksel bazında maksimumla birleştirmek günlük tespit şansını kabaca ikiye katlar. Güven
sınıfı `FIRE_CONFIDENCE` altındaki pikseller — su, bulut ve işlenmemiş sınıfları dâhil —
yangın dışı sayılır.

### Temporal compositing / Zamansal derleme

`MOD13Q1` has a 16-day revisit, so the most recent composite within a 32-day lookback is
used. `MOD11A1` is daily but frequently cloud-obscured, so a 3-day mean is taken. This
follows Section 2.3 of the manuscript and is unchanged in this revision.

`MOD13Q1` 16 günlük tekrar ziyaret süresine sahiptir; bu yüzden 32 günlük geriye bakış
içindeki en yeni kompozit alınır. `MOD11A1` günlüktür ama sık sık bulut altında kalır,
bu yüzden 3 günlük ortalama alınır. Bu, makalenin 2.3 bölümüne uygundur ve bu sürümde
değişmemiştir.

In [ ]:
def daily_fire_mask(date):
    '''Binary active-fire mask for one day, Terra and Aqua merged.

    Ikili aktif yangin maskesi (Terra + Aqua birlesik), tek gun icin.
    '''
    d0 = ee.Date(date)
    d1 = d0.advance(1, 'day')
    terra = ee.ImageCollection('MODIS/061/MOD14A1').filterDate(d0, d1).select('FireMask')
    aqua  = ee.ImageCollection('MODIS/061/MYD14A1').filterDate(d0, d1).select('FireMask')
    col   = terra.merge(aqua)
    fm = ee.Image(ee.Algorithms.If(col.size().gt(0),
                                   col.max(),
                                   ee.Image.constant(0).rename('FireMask')))
    return fm.gte(FIRE_CONFIDENCE).rename('fire').unmask(0).toFloat().clip(REGION)


def feature_stack(date):
    '''The 13 environmental driver bands for one day (fire mask excluded).

    Bir gune ait 13 cevresel surucu bandi (yangin maskesi haric).

    Bands remain MASKED where no valid observation exists; the mask is consumed by
    build_sample_image() to derive the `valid` band and only then replaced with zero.
    Gecerli gozlem olmayan yerlerde bantlar MASKELI kalir; bu maske build_sample_image()
    tarafindan `valid` bandini turetmek icin kullanilir ve ancak ondan sonra sifirlanir.
    '''
    d0 = ee.Date(date)
    d1 = d0.advance(1, 'day')

    # --- Vegetation / Bitki ortusu -------------------------------------------------
    # Most recent 16-day composite within a 32-day lookback.
    ndvi = (ee.ImageCollection('MODIS/061/MOD13Q1')
              .filterDate(d0.advance(-32, 'day'), d1).select('NDVI')
              .sort('system:time_start', False).first()
              .multiply(0.0001).rename('ndvi'))

    # --- Land surface temperature / Arazi yuzey sicakligi ---------------------------
    # 3-day mean fills cloud gaps. Scale factor 0.02, Kelvin -> Celsius.
    lst = (ee.ImageCollection('MODIS/061/MOD11A1')
             .filterDate(d0.advance(-3, 'day'), d1).select('LST_Day_1km')
             .mean().multiply(0.02).subtract(273.15).rename('lst'))

    # --- Meteorology / Meteoroloji --------------------------------------------------
    # ERA5-Land is published with a short delay; fall back to the most recent record
    # within a 7-day window so that recent dates remain processable.
    # ERA5-Land kisa bir gecikmeyle yayimlanir; guncel tarihlerin islenebilir kalmasi
    # icin 7 gunluk pencere icindeki en yeni kayda dusulur.
    era = ee.Image(ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
                     .filterDate(d0.advance(-7, 'day'), d1)
                     .sort('system:time_start', False).first())

    air = era.select('temperature_2m').subtract(273.15).rename('air_temp')
    dew = era.select('dewpoint_temperature_2m').subtract(273.15)

    # Relative humidity from the Magnus relation, clipped to a physical range.
    # Magnus bagintisiyla bagil nem, fiziksel araliga kirpilmis.
    a = dew.multiply(17.625).divide(dew.add(243.04))
    b = air.multiply(17.625).divide(air.add(243.04))
    humidity = a.subtract(b).exp().multiply(100).clamp(0, 100).rename('humidity')

    # Vapour pressure deficit: the drying power of the air, in kPa. Tetens equation
    # for saturation vapour pressure, evaluated at air and dew-point temperature.
    # VPD predicts fine-fuel moisture more directly than relative humidity does,
    # because the same RH is far more desiccating at 35 degC than at 15 degC.
    # Buhar basinci acigi: havanin kurutma gucu, kPa. Doyma buhar basinci icin
    # Tetens denklemi, hava ve ciglenme sicakliginda hesaplanir. VPD, ince yakit
    # nemini bagil nemden daha dogrudan yordar; ayni bagil nem 35 derecede
    # 15 dereceye gore cok daha kurutucudur.
    def _es(t):
        return t.multiply(17.27).divide(t.add(237.3)).exp().multiply(0.6108)
    vpd = _es(air).subtract(_es(dew)).max(0).rename('vpd')

    u     = era.select('u_component_of_wind_10m').rename('wind_u')
    v     = era.select('v_component_of_wind_10m').rename('wind_v')
    speed = u.hypot(v).rename('wind_speed')
    soil  = era.select('volumetric_soil_water_layer_1').rename('soil_moisture')

    # --- Precipitation / Yagis ------------------------------------------------------
    # CHIRPS is the primary source. It lags real time by several weeks, so ERA5-Land
    # total precipitation (m -> mm) is substituted when CHIRPS has no record.
    # Birincil kaynak CHIRPS'tir. Gercek zamana gore haftalarca gecikir; kayit yoksa
    # ERA5-Land toplam yagisi (m -> mm) kullanilir.
    def _precip_sum(days, name):
        '''Accumulated precipitation over the preceding `days`, CHIRPS with ERA5 fallback.'''
        c = (ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
               .filterDate(d0.advance(-days, 'day'), d1).select('precipitation'))
        e = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
               .filterDate(d0.advance(-days, 'day'), d1)
               .select('total_precipitation_sum'))
        return ee.Image(ee.Algorithms.If(
            c.size().gt(0), c.sum(),
            ee.Image(ee.Algorithms.If(e.size().gt(0), e.sum().multiply(1000),
                                      ee.Image.constant(0))))).rename(name)

    # Antecedent precipitation. The 24 h total is zero on 91.5 % of pixels in the v2
    # archive and therefore carries almost no information on its own; 7-day and
    # 30-day accumulations encode the drought state that actually governs fuel
    # dryness and are among the strongest predictors in operational fire-weather
    # indices such as the FWI.
    # Onceki yagis. 24 saatlik toplam, v2 arsivinde piksellerin %91.5'inde sifirdir
    # ve tek basina neredeyse hic bilgi tasimaz; 7 ve 30 gunluk birikimler yakit
    # kuruluugunu asil belirleyen kuraklik durumunu kodlar ve FWI gibi operasyonel
    # yangin-hava indekslerinin en guclu yordayicilari arasindadir.
    precip     = _precip_sum(1,  'precip')
    precip_7d  = _precip_sum(7,  'precip_7d')
    precip_30d = _precip_sum(30, 'precip_30d')

    # --- Terrain / Arazi ------------------------------------------------------------
    terr   = ee.Terrain.products(ee.Image('USGS/SRTMGL1_003'))
    elev   = terr.select('elevation').rename('elevation')
    slope  = terr.select('slope').rename('slope')
    aspect = terr.select('aspect').rename('aspect')

    # --- Fuel type / Yakit tipi -----------------------------------------------------
    # MCD12Q1 is annual and trails the current year; fall back to the latest available.
    # MCD12Q1 yilliktir ve gecerli yilin gerisinde kalir; en yeni mevcut yila dusulur.
    yr     = d0.get('year')
    lc_col = ee.ImageCollection('MODIS/061/MCD12Q1').select('LC_Type1')
    lc_yr  = lc_col.filter(ee.Filter.calendarRange(yr, yr, 'year'))
    lc = ee.Image(ee.Algorithms.If(lc_yr.size().gt(0),
                                   lc_yr.first(),
                                   lc_col.sort('system:time_start', False).first())
                  ).rename('landcover')

    return (ndvi.addBands([lst, air, humidity, vpd, speed, u, v,
                           precip, precip_7d, precip_30d, soil,
                           elev, slope, aspect, lc])
                .toFloat().clip(REGION))

### The `valid` band / `valid` bandı

`valid` is the pixel-wise minimum of the 13 environmental band masks, evaluated **before**
`unmask(0)`. It is 1 only where every input variable was genuinely observed, and 0 where a
value was fabricated — outside the national border, or where cloud removed a retrieval.

Measurement of the previous archive showed an identical 15 % zero rate across all
environmental bands, confirming that these fabricated zeros were being presented to the
network as if they were physical measurements. A relative humidity of 0 % and an
unobserved pixel are numerically indistinguishable without this band.

`valid`, 13 çevresel bant maskesinin piksel bazında minimumudur ve `unmask(0)`
uygulanmadan **önce** hesaplanır. Yalnızca her girdi değişkeninin gerçekten gözlendiği
yerde 1, bir değerin uydurulduğu yerde 0'dır — ülke sınırı dışında veya bulutun ölçümü
engellediği yerlerde.

Önceki arşivin ölçümü, tüm çevresel bantlarda birebir aynı %15 sıfır oranı gösterdi; bu
da uydurulmuş sıfırların ağa fiziksel ölçümmüş gibi sunulduğunu doğruladı. Bu bant
olmadan %0 bağıl nem ile gözlenmemiş bir piksel sayısal olarak ayırt edilemez.

In [ ]:
def build_sample_image(date):
    '''17-band export stack for one day, reprojected onto the common grid.

    Bir gune ait 17 bantli disa aktarim yigini, ortak gride yeniden projekte edilmis.

    Band order is fixed by ALL_BANDS and is contractual with the training pipeline.
    Bant sirasi ALL_BANDS ile sabitlenir ve egitim hattiyla sozlesmeseldir.
    '''
    d0 = ee.Date(date)
    feats = feature_stack(d0)

    fire_t  = daily_fire_mask(d0).rename('fire')
    fire_n1 = daily_fire_mask(d0.advance(1, 'day')).rename('fire_next')
    fire_n2 = daily_fire_mask(d0.advance(2, 'day')).rename('fire_next2')

    # Temporal context. A single day's detection cannot distinguish a fire that has
    # burned continuously for three days from an isolated thermal anomaly, yet the
    # two behave completely differently on day t+1. Supplying t-1 and t-2 gives the
    # network that distinction without moving to a recurrent architecture.
    # Zamansal baglam. Tek gunluk bir tespit, uc gundur kesintisiz yanan bir yangini
    # tek seferlik bir termal anomaliden ayirt edemez; oysa ikisi t+1 gununde
    # tamamen farkli davranir. t-1 ve t-2'yi vermek, yinelemeli bir mimariye
    # gecmeden aga bu ayrimi kazandirir.
    fire_p1 = daily_fire_mask(d0.advance(-1, 'day')).rename('fire_prev1')
    fire_p2 = daily_fire_mask(d0.advance(-2, 'day')).rename('fire_prev2')

    # Observation validity, taken from the band masks before they are filled.
    # Gozlem gecerliligi, bantlar doldurulmadan once maskelerinden alinir.
    valid = feats.mask().reduce(ee.Reducer.min()).rename('valid').toFloat()

    stack = (feats.addBands([fire_p2, fire_p1, fire_t, fire_n1, fire_n2, valid])
                  .select(ALL_BANDS))

    # unmask(sameFootprint=False) makes the image defined everywhere, which guarantees
    # that neighbourhoodToArray always returns a full 65 x 65 array. Without it, patches
    # near a mask edge are truncated and fail to parse during training.
    # unmask(sameFootprint=False) goruntuyu her yerde tanimli yapar; boylece
    # neighborhoodToArray her zaman tam 65 x 65 dizi dondurur. Aksi halde maske kenarina
    # yakin yamalar kisalir ve egitimde cozumlenemez.
    return stack.unmask(0, False).reproject(PROJ)

## 5 — Sampling and export / Örnekleme ve dışa aktarım

Up to `MAX_POINTS_PER_DAY` burning pixels are drawn per fire day with
`stratifiedSample`, which prevents a single very large event from dominating the archive.

**Implementation note.** `stratifiedSample` attaches a scalar property named `fire` to
each sampled point. That property collides with the 65 × 65 `fire` *band* produced by
`neighborhoodToArray` and silently reduces it to a 1 × 1 scalar, which corrupts the
channel. All point properties are therefore discarded and only the geometry is retained;
longitude and latitude are re-attached under names that cannot collide with any band.

Her yangın gününde `stratifiedSample` ile en fazla `MAX_POINTS_PER_DAY` yanan piksel
seçilir; bu, tek bir çok büyük olayın arşive hâkim olmasını engeller.

**Uygulama notu.** `stratifiedSample`, örneklenen her noktaya `fire` adlı skaler bir
özellik ekler. Bu özellik, `neighborhoodToArray`'in ürettiği 65 × 65 `fire` *bandıyla*
çakışır ve onu sessizce 1 × 1 skalere indirger; kanal bozulur. Bu yüzden noktaların tüm
özellikleri atılır ve yalnızca geometri korunur; boylam ve enlem, hiçbir bantla
çakışmayacak adlarla yeniden eklenir.

In [ ]:
def fire_points(date, max_points):
    '''Sample burning pixels for one day, returning geometry-only features.'''
    fire_t = daily_fire_mask(date).selfMask().toInt()
    pts = fire_t.stratifiedSample(
        numPoints=max_points, classBand='fire',
        region=REGION.geometry(), scale=SCALE, projection=PROJ,
        seed=SEED, geometries=True, dropNulls=True)
    # Drop every property; keep geometry. Re-attach coordinates under safe names.
    # Tum ozellikleri at, geometriyi tut. Koordinatlari guvenli adlarla yeniden ekle.
    return pts.map(lambda f: ee.Feature(f.geometry()).set(
        'lon', f.geometry().coordinates().get(0),
        'lat', f.geometry().coordinates().get(1)))


def export_day(date_str):
    '''Submit one day's patches to Drive as a gzip TFRecord shard.'''
    date  = ee.Date(date_str)
    pts   = fire_points(date, MAX_POINTS_PER_DAY)
    stack = build_sample_image(date)

    arrays  = stack.neighborhoodToArray(ee.Kernel.square(PATCH_RADIUS, 'pixels'))
    samples = arrays.sampleRegions(collection=pts, scale=SCALE,
                                   projection=PROJ, geometries=False)
    samples = samples.map(lambda f: f.set('date', date_str))

    name = 'firespread_' + date_str.replace('-', '')
    task = ee.batch.Export.table.toDrive(
        collection=samples,
        description=name,
        folder=DRIVE_FOLDER,
        fileNamePrefix=name,
        fileFormat='TFRecord',
        selectors=ALL_BANDS + META_COLS)
    task.start()
    return task

## 6 — Day selection / Gün seçimi

A day is exported only if Türkiye recorded at least `MIN_FIRE_PIXELS` active fire pixels.
Testing that one day at a time would cost roughly 1100 round trips to the Earth Engine
servers, so the counts are batched.

**How the batching must be done.** The obvious approach — mapping `reduceRegion` over a
list of dates — opens *one aggregation per day* and immediately trips Earth Engine's
`Too many concurrent aggregations` limit; the client then retries into the same wall and
the whole span fails. Instead each day is made a separate **band** of a single image, so
one `reduceRegion` call returns every count at once. That is one aggregation per chunk
rather than one per day.

If a chunk still fails, it is bisected and each half retried, down to single days, so a
single problematic date cannot discard a whole span.

Bir gün, yalnızca Türkiye'de en az `MIN_FIRE_PIXELS` aktif yangın pikseli kaydedildiyse
dışa aktarılır. Bunu günlük test etmek Earth Engine sunucularına yaklaşık 1100 gidiş-dönüş
demektir, bu yüzden sayımlar gruplanır.

**Gruplamanın nasıl yapılması gerektiği.** Akla ilk gelen yaklaşım — `reduceRegion`'ı bir
tarih listesi üzerinde map etmek — *gün başına bir toplulaştırma* açar ve anında Earth
Engine'in `Too many concurrent aggregations` sınırına takılır; istemci de aynı duvara
yeniden denemeler yapar ve tüm aralık başarısız olur. Bunun yerine her gün tek bir
görüntünün ayrı bir **bandı** yapılır; böylece tek bir `reduceRegion` çağrısı tüm sayımları
birden döndürür. Bu, gün başına değil grup başına bir toplulaştırmadır.

Bir grup yine de başarısız olursa ikiye bölünür ve her yarısı yeniden denenir; tek bir
sorunlu tarih tüm aralığı çöpe atamaz.

In [ ]:
import time

CHUNK_DAYS = 10   # days per aggregation / toplulastirma basina gun sayisi

# ─────────────────── DOCUMENTED SENSOR OUTAGES / BELGELENMIS SENSOR KESINTILERI ──
# Days on which Terra MODIS did not acquire data cannot produce a valid `lst` or
# `ndvi` channel: MOD11A1 has a 3-day compositing window, so a window falling entirely
# inside an outage yields an empty ImageCollection, and .mean() on it returns a
# band-less image that fails at .rename(). Such days are excluded rather than forced
# through, because filling them from stale observations would fabricate two of the
# fourteen input channels.
#
# Terra MODIS'in veri toplamadigi gunler gecerli bir `lst` veya `ndvi` kanali
# uretemez: MOD11A1 3 gunluk bir derleme penceresi kullanir; pencere tamamen kesinti
# icine duserse ImageCollection bos kalir, .mean() bantsiz bir goruntu dondurur ve
# .rename() hata verir. Bu gunler zorlanmak yerine dislanir, cunku onlari eski
# gozlemlerle doldurmak on dort girdi kanalindan ikisini uydurmak olurdu.
#
# Reference / Kaynak: LP DAAC, "Terra Constellation Exit & Data Outage,
# October 10-19, 2022". Acquisition ceased 10 Oct 2022 for retrograde manoeuvres on
# 12 and 19 Oct; instruments were still recovering through 21 Oct.
KNOWN_OUTAGES = [
    (datetime.date(2022, 10, 10), datetime.date(2022, 10, 22)),
]


def in_known_outage(day):
    return any(lo <= day <= hi for lo, hi in KNOWN_OUTAGES)


def _transient(exc):
    '''True if an Earth Engine error is a capacity limit worth retrying.'''
    m = str(exc).lower()
    return any(k in m for k in
               ('concurrent', 'too many', 'quota', 'timed out', 'try again', 'backend'))


def chunk_fire_counts(first_day, n_days):
    '''Daily active-fire pixel counts for a short span, in ONE server aggregation.

    Gunluk aktif yangin piksel sayilari, TEK sunucu toplulastirmasiyla.

    Each day becomes a separate BAND of one image, so a single reduceRegion returns
    every count together. Mapping reduceRegion over a list of dates instead opens one
    aggregation per day and trips the 'Too many concurrent aggregations' limit.
    Her gun tek bir goruntunun ayri bir BANDI olur; boylece tek reduceRegion tum
    sayimlari birlikte dondurur. reduceRegion'i bir tarih listesi uzerinde map etmek
    gun basina bir toplulastirma acar ve 'Too many concurrent aggregations' sinirina
    takilir.
    '''
    start = ee.Date(first_day.isoformat())
    stack = ee.Image.cat([daily_fire_mask(start.advance(i, 'day')).rename(f'd{i:02d}')
                          for i in range(n_days)])

    last = None
    for attempt in range(4):
        try:
            res = stack.reduceRegion(
                reducer=ee.Reducer.sum(), geometry=REGION.geometry(), scale=SCALE,
                maxPixels=1e10, bestEffort=False, tileScale=8).getInfo()
            return [(first_day + datetime.timedelta(days=i), res.get(f'd{i:02d}') or 0)
                    for i in range(n_days)]
        except Exception as exc:
            last = exc
            if not _transient(exc):
                raise
            wait = 8 * (2 ** attempt)
            print(f'      capacity limit, waiting {wait}s / kapasite siniri, {wait}s bekleniyor')
            time.sleep(wait)
    raise last


def counts_for_span(first_day, n_days, depth=0):
    '''Counts for a span, bisecting on failure so one bad date cannot lose the rest.'''
    try:
        return chunk_fire_counts(first_day, n_days)
    except Exception as exc:
        if n_days <= 1 or depth >= 5:
            print(f'      {first_day}: giving up / vazgecildi ({str(exc)[:60]})')
            return []
        half = n_days // 2
        print(f'      {first_day} +{n_days}d failed, bisecting / bolunuyor')
        return (counts_for_span(first_day, half, depth + 1)
                + counts_for_span(first_day + datetime.timedelta(days=half),
                                  n_days - half, depth + 1))


def season_bounds(year):
    '''First and last calendar day of the fire season, clipped to available data.'''
    first = datetime.date(year, FIRE_SEASON[0], 1)
    m = FIRE_SEASON[1]
    last = (datetime.date(year + 1, 1, 1) if m == 12
            else datetime.date(year, m + 1, 1)) - datetime.timedelta(days=1)
    return first, min(last, LAST_SAMPLE_DATE)


def candidate_days():
    '''All fire-season days with sufficient activity, within the available archive.'''
    out = []
    for year in YEARS:
        first, last = season_bounds(year)
        if first > last:
            print(f'  {year}: outside available archive / mevcut arsivin disinda')
            continue
        kept_year, scanned, excluded = 0, 0, 0
        day = first
        while day <= last:
            n = min(CHUNK_DAYS, (last - day).days + 1)
            counts = counts_for_span(day, n)
            scanned += len(counts)
            hits = [d for d, c in counts if c >= MIN_FIRE_PIXELS]
            usable = [d for d in hits if not in_known_outage(d)]
            excluded += len(hits) - len(usable)
            out.extend(usable)
            kept_year += len(usable)
            day += datetime.timedelta(days=n)
        note = f'  ({excluded} excluded, sensor outage / sensor kesintisi)' if excluded else ''
        print(f'  {year}: {kept_year:>3} fire days of {scanned} scanned '
              f'/ {scanned} gunde {kept_year} yangin gunu{note}')
    return sorted(d.isoformat() for d in out)


print('Scanning fire seasons / yangin sezonlari taraniyor ...\n')
DAYS = candidate_days()
print(f'\nTotal candidate fire days / toplam aday yangin gunu: {len(DAYS)}')
if DAYS:
    print(f'Range / aralik: {DAYS[0]} .. {DAYS[-1]}')
else:
    print('No days selected. Check the errors above before continuing.')
    print('Hic gun secilmedi. Devam etmeden once yukaridaki hatalari kontrol edin.')

## 7 — Resumable export / Yeniden başlatılabilir dışa aktarım

Generating the full archive takes longer than a single Colab session. Days whose shard is
already present in Drive, or whose export task is already queued or running, are skipped.
Re-running this notebook therefore continues from where the previous session stopped.

Tam arşivi üretmek tek bir Colab oturumundan uzun sürer. Parçası Drive'da hâlihazırda
bulunan veya dışa aktarım görevi zaten kuyruğa alınmış ya da çalışan günler atlanır. Bu
nedenle bu notebook'u yeniden çalıştırmak, önceki oturumun bıraktığı yerden devam eder.

In [ ]:
import os, re

def completed_days():
    '''Days already exported to Drive, or currently queued / running in Earth Engine.'''
    done = set()

    # (a) Shards already written to Drive. / Drive'a yazilmis parcalar.
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        folder = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
        if os.path.isdir(folder):
            for fn in os.listdir(folder):
                m = re.match(r'firespread_(\d{8})', fn)
                if m:
                    s = m.group(1)
                    done.add(f'{s[:4]}-{s[4:6]}-{s[6:]}')
            print(f'Drive: {len(done)} shard(s) already present / parca zaten mevcut')
        else:
            print(f'Drive: folder {DRIVE_FOLDER} not found yet / klasor henuz yok')
    except Exception as exc:
        print(f'Drive scan skipped / Drive taramasi atlandi: {exc}')

    # (b) Tasks already submitted this or a previous session.
    #     Bu veya onceki oturumda gonderilmis gorevler.
    try:
        active = 0
        for op in ee.data.listOperations():
            meta = op.get('metadata', {})
            state = meta.get('state', '')
            desc  = meta.get('description', '')
            m = re.match(r'firespread_(\d{8})$', desc)
            if m and state in ('PENDING', 'RUNNING', 'SUCCEEDED'):
                s = m.group(1)
                done.add(f'{s[:4]}-{s[4:6]}-{s[6:]}')
                if state in ('PENDING', 'RUNNING'):
                    active += 1
        print(f'Earth Engine: {active} task(s) pending or running / bekleyen veya calisan')
    except Exception as exc:
        print(f'Task scan skipped / gorev taramasi atlandi: {exc}')

    return done


DONE      = completed_days()
REMAINING = [d for d in DAYS if d not in DONE]

print(f'\nCompleted / tamamlanan : {len(DAYS) - len(REMAINING)}')
print(f'Remaining / kalan      : {len(REMAINING)}')
print(f'This run / bu calistirma: {min(len(REMAINING), SUBMIT_LIMIT)} '
      f'(SUBMIT_LIMIT = {SUBMIT_LIMIT})')

In [ ]:
submitted = []
for day in REMAINING[:SUBMIT_LIMIT]:
    try:
        export_day(day)
        submitted.append(day)
        print(f'  {day}  submitted / gonderildi')
    except Exception as exc:
        print(f'  {day}  FAILED / BASARISIZ: {exc}')

print(f'\n{len(submitted)} export task(s) submitted to Drive/{DRIVE_FOLDER}.')
print(f'{len(REMAINING) - len(submitted)} day(s) still remaining / gun hala kaldi.')
if len(REMAINING) > len(submitted):
    print('Re-run cells 7 and 8 to continue. / Devam icin 7 ve 8. hucreleri tekrar calistirin.')
print('\nMonitor: https://code.earthengine.google.com/tasks')

## 8 — Progress monitoring / İlerleme takibi

Re-run this cell periodically. Earth Engine processes a limited number of tasks
concurrently, so a large batch will complete over several hours.

Bu hücreyi düzenli aralıklarla yeniden çalıştırın. Earth Engine aynı anda sınırlı sayıda
görev işler; büyük bir grup birkaç saat içinde tamamlanır.

In [ ]:
import re
from collections import Counter

states = Counter()
failures = []
for op in ee.data.listOperations():
    meta = op.get('metadata', {})
    if not re.match(r'firespread_\d{8}$', meta.get('description', '')):
        continue
    st = meta.get('state', 'UNKNOWN')
    states[st] += 1
    if st == 'FAILED':
        # The failure reason lives at the TOP level of the operation, not inside
        # `metadata`; reading meta['error'] silently yields an empty string.
        # Hata nedeni islemin UST duzeyindedir, `metadata` icinde degil;
        # meta['error'] okumak sessizce bos dize dondurur.
        err = (op.get('error') or {}).get('message') or '(no message returned)'
        failures.append((meta.get('description'), err))

print('Export task states / disa aktarim gorev durumlari')
print('-' * 50)
for st, n in sorted(states.items()):
    print(f'  {st:<12} {n:>5}')

if failures:
    print(f'\n{len(failures)} failed task(s) / basarisiz gorev — first 10:')
    for desc, msg in failures[:10]:
        print(f'  {desc}: {msg[:110]}')
    print('\nFailed days are not recorded as complete and will be retried '
          'on the next run of cell 8.')
    print('Basarisiz gunler tamamlanmis sayilmaz; 8. hucrenin bir sonraki '
          'calistirilmasinda yeniden denenir.')

## 9 — Retrieving the archive / Arşivi indirme

> **Keep each archive version in its own directory.** A v3 shard has 19 input bands;
> a v2 shard has 14. Mixing them in one directory makes the channel axis inconsistent
> and the loader will reject the short records.
>
> **Her arşiv sürümünü kendi dizininde tutun.** Bir v3 parçası 19 girdi bandı taşır,
> bir v2 parçası 14. Aynı dizinde karıştırmak kanal eksenini tutarsız hâle getirir ve
> yükleyici kısa kayıtları reddeder.

When every task has reached `SUCCEEDED`, download the contents of
Drive → `GEE_FireSpread_v3/` into `data/spread_v3/`:

Tüm görevler `SUCCEEDED` durumuna ulaştığında, Drive → `GEE_FireSpread_v3/` klasörünün
içeriğini `data/spread_v3/` dizinine indirin:

```bash
mkdir -p data/spread_v3     # v3 shards live here / v3 parçaları buraya
# data/spread/    stays as the v2 archive / v2 arşivi olarak kalır
```

Then convert to a memory-mapped local cache:

Ardından belleğe eşlenmiş yerel önbelleğe dönüştürün:

```bash
python src/tfrecord_to_npy.py --verify   # integrity report / bütünlük raporu
python src/train.py                      # train the U-Net / U-Net'i eğit
```

The conversion step is not optional. It is what allows the training loop to read patches
without decompressing gzip on every epoch, and it produces the per-channel normalisation
statistics from the training split alone.

Dönüştürme adımı isteğe bağlı değildir. Eğitim döngüsünün her epoch'ta gzip açmadan yama
okumasını sağlayan ve kanal başına normalizasyon istatistiklerini yalnızca eğitim
bölmesinden üreten adım budur.

## 10 — Visual inspection / Görsel inceleme

A sanity check on a single day: today's detections in red, tomorrow's in yellow. The
spatial offset between them is the signal the model is asked to learn.

Tek bir gün üzerinde akıl sağlığı kontrolü: bugünün tespitleri kırmızı, yarınınkiler
sarı. Aralarındaki uzamsal kayma, modelden öğrenmesi istenen sinyaldir.

In [ ]:
# 2021-07-29: second day of the Manavgat fire. / Manavgat yanginlarinin ikinci gunu.
DEMO_DATE = '2021-07-29'

fire_t    = daily_fire_mask(DEMO_DATE)
fire_next = daily_fire_mask(ee.Date(DEMO_DATE).advance(1, 'day'))
feats     = feature_stack(DEMO_DATE)
valid     = feats.mask().reduce(ee.Reducer.min())

m = geemap.Map()
m.centerObject(REGION, 6)
m.addLayer(feats.select('ndvi'),
           {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}, 'NDVI')
m.addLayer(feats.select('wind_speed'),
           {'min': 0, 'max': 12, 'palette': ['white', 'blue', 'purple']}, 'Wind speed')
m.addLayer(valid, {'min': 0, 'max': 1, 'palette': ['black', 'white']},
           'Valid observations', False)
m.addLayer(fire_t.selfMask(),    {'palette': ['red']},    'Fire, day t')
m.addLayer(fire_next.selfMask(), {'palette': ['yellow']}, 'Fire, day t+1')
m